<a href="https://colab.research.google.com/github/ttobias28/PhishRadar-Project/blob/main/PhishRadar_tfidf_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhishRadar — Approach 1: TF-IDF + Logistic Regression for phishing detection

**Authors:** Connor Buttrey, Oliver Conley, Robert Morriss, Ashley Smutnik, Teagan Tobias

<br>

**Data: the team's cross-source splits** (`phishradar_splits.zip`). The model trains on **Enron, Ling and SpamAssassin** only. It is then tested on:

| Test set | What it tells you |
|---|---|
| `test_insource` | Same sources as training. The "normal" score. |
| `test_ceas` | A source it never saw (spam + legit). Does it generalize? |
| `test_nazario_paired` | Real phishing (Nazario) mixed 50/50 with unseen legit emails (CEAS). The closest thing to a real phishing test. |
| `test_nazario` | Real phishing only. Shows the detection rate (recall). |
| `test_nigerian` | Advance-fee scams only. Shows the detection rate (recall). |

**Every time (Google Colab):**

1. Open this notebook in Colab (colab.research.google.com → **File → Upload notebook**).
2. No GPU is needed for this model. The default runtime is fine.
3. Make sure the split CSVs are in your Google Drive at `MyDrive/phishradar/data/splits/`.
4. Run the cells top to bottom with **Shift+Enter** (or **Runtime → Run all**). The first cell asks to connect to Google Drive: click **Connect** and approve.
5. Results are saved to `MyDrive/phishradar/results/` in your Drive, so they are still there after the session ends.

**Running in plain Jupyter instead?** Put the CSVs in `data/splits/` next to this notebook. The notebook detects that it isn't in Colab and skips the Google Drive step.

**Run time:** about a minute. Training uses all ~30,000 training emails.

## 1. Setup: connect to your files

In [1]:
from pathlib import Path
import sklearn; print('scikit-learn', sklearn.__version__)

try:  # running in Google Colab
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/phishradar')
except ImportError:  # running locally (VS Code / Jupyter): use the current folder
    BASE = Path('.')

SPLITS = BASE / 'data' / 'splits'
OUT = BASE / 'results' / 'predictions'
OUT.mkdir(parents=True, exist_ok=True)
print('Reading data from:', SPLITS.resolve())
print('Files found:', sorted(p.name for p in SPLITS.glob('*.csv')))

scikit-learn 1.6.1
Mounted at /content/drive
Reading data from: /content/drive/MyDrive/phishradar/data/splits
Files found: ['test_ceas.csv', 'test_insource.csv', 'test_nazario.csv', 'test_nazario_paired.csv', 'test_nigerian.csv', 'train.csv', 'val.csv']


## 2. Load the data
The model only learns from `train.csv`. `val.csv` is for choosing settings. The `test_*` files are only used at the end.

In [2]:
import numpy as np
import pandas as pd

SEED = 42
TEST_FILES = ['test_insource', 'test_ceas', 'test_nazario', 'test_nigerian', 'test_nazario_paired']

train = pd.read_csv(SPLITS / 'train.csv')
val = pd.read_csv(SPLITS / 'val.csv')
print(len(train), 'train emails,', len(val), 'val emails')
train.head()

30217 train emails, 3778 val emails


,text,label,source
0,cheap oem software shipping worldwide . worryi...,1,Enron
1,Re: Wifi query I think what you're looking at ...,0,SpamAssassin
2,nels-29 final announcement final announcement ...,0,Ling
3,"coca cola , mbna america , nascar partner with...",1,Enron
4,hi 325 and hi 368 revision effective 6 / 10 / ...,0,Enron


## 3. Turn emails into numbers (TF-IDF)
The vectorizer learns its vocabulary from **training text only**. It never sees val or test text.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=100_000)
X_train = vec.fit_transform(train['text'])
X_val = vec.transform(val['text'])
print('vocabulary size:', len(vec.vocabulary_))

vocabulary size: 100000


## 4. Train, trying a few settings
`C` controls how strongly the model fits the training data. We keep whichever `C` scores best on the validation set.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

best = None
for C in [0.1, 1, 10, 100]:
    model = LogisticRegression(C=C, max_iter=2000, random_state=SEED)
    model.fit(X_train, train['label'])
    f1 = f1_score(val['label'], model.predict(X_val))
    print(f'C={C:<6} validation F1 = {f1:.4f}')
    if best is None or f1 > best[0]:
        best = (f1, C, model)

_, best_C, model = best
print('Best C:', best_C)

C=0.1    validation F1 = 0.9766
C=1      validation F1 = 0.9861
C=10     validation F1 = 0.9886
C=100    validation F1 = 0.9880
Best C: 10


## 5. Predict every test file and save the predictions
Each file saved is in the team format: `row_id, y_true, y_pred, score`.

In [5]:
from sklearn.metrics import precision_score, recall_score

def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    has_legit = (y == 0).any()   # Nazario / Nigerian have no legitimate emails
    return {
        'rows': len(y),
        'recall': recall_score(y, p, zero_division=0),
        'precision': precision_score(y, p, zero_division=0) if has_legit else np.nan,
        'f1': f1_score(y, p, zero_division=0) if has_legit else np.nan,
        'fpr': ((p == 1) & (y == 0)).sum() / (y == 0).sum() if has_legit else np.nan,
    }

rows = []
for name in ['val'] + TEST_FILES:
    df = val if name == 'val' else pd.read_csv(SPLITS / f'{name}.csv')
    X = X_val if name == 'val' else vec.transform(df['text'])
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    if name != 'val':
        pd.DataFrame({'row_id': range(len(df)), 'y_true': df['label'],
                      'y_pred': preds, 'score': probs}
                     ).to_csv(OUT / f'tfidf_{name}.csv', index=False)
    rows.append({'set': name, **metrics(df['label'], preds)})

results = pd.DataFrame(rows).round(4)
results.to_csv(BASE / 'results' / 'tfidf_results.csv', index=False)
results

,set,rows,recall,precision,f1,fpr
0,val,3778,0.9880,0.9893,0.9886,0.0078
1,test_insource,3778,0.9912,0.9887,0.9899,0.0082
2,test_ceas,38748,0.8500,0.9218,0.8845,0.0896
3,test_nazario,1544,0.7902,NaN,NaN,NaN
4,test_nigerian,3243,0.9935,NaN,NaN,NaN
5,test_nazario_paired,3088,0.7902,0.8931,0.8385,0.0946


**Reading the table:** `recall` = share of real phishing/scam caught. `precision` = share of flagged emails that were really phishing. `fpr` = share of legitimate emails wrongly flagged. Nazario and Nigerian have no legitimate emails, so only recall is meaningful there (shown as NaN for the rest).

## 6. Which words does the model rely on?

In [6]:
words = np.array(vec.get_feature_names_out())
order = np.argsort(model.coef_[0])
top = pd.concat([
    pd.DataFrame({'direction': 'phishing', 'word': words[order[::-1][:30]], 'weight': model.coef_[0][order[::-1][:30]]}),
    pd.DataFrame({'direction': 'legitimate', 'word': words[order[:30]], 'weight': model.coef_[0][order[:30]]}),
])
top.to_csv(BASE / 'results' / 'tfidf_top_words.csv', index=False)
print('Top phishing words:  ', ', '.join(top[top.direction == 'phishing'].word.head(15)))
print('Top legitimate words:', ', '.join(top[top.direction == 'legitimate'].word.head(15)))

Top phishing words:   your, 2004, 2005, remove, you, money, site, our, here, mobile, no, hello, life, viagra, software
Top legitimate words: enron, vince, thanks, 2002, the, louise, attached, 2001, wrote, 713, 2000, language, doc, gas, edu


**Look at these words.** If you see names like `enron` or years like `2004`, the model is partly recognizing *which dataset* an email came from, not just phishing. Worth discussing in the report.

## 7. Save the trained model
Saves the fitted vectorizer and model so other experiments (for example the "AI polish" test) can score new emails without retraining. They go in a `models/` folder next to `results/`. **Do not commit these files to GitHub**: they are learned from email text.

*Sections 7-9 are extras and add a few minutes of run time (section 8 retrains the model four times).*

In [7]:
import joblib

MODELS = BASE / 'models'
MODELS.mkdir(parents=True, exist_ok=True)
joblib.dump(vec, MODELS / 'tfidf_vectorizer.joblib')
joblib.dump(model, MODELS / 'tfidf_model.joblib')

def predict_texts(texts):
    """Score any list of email texts. Returns (0/1 predictions, phishing probabilities)."""
    probs = model.predict_proba(vec.transform(texts))[:, 1]
    return (probs >= 0.5).astype(int), probs

# Sanity check: reload from disk and confirm the predictions are identical
vec2 = joblib.load(MODELS / 'tfidf_vectorizer.joblib')
model2 = joblib.load(MODELS / 'tfidf_model.joblib')
check = pd.read_csv(SPLITS / 'test_nazario.csv')['text'].head(200)
assert (model2.predict(vec2.transform(check)) == predict_texts(check)[0]).all()
print('Saved to', MODELS.resolve(), '- reload check passed')

Saved to /content/drive/MyDrive/phishradar/models - reload check passed


## 8. Leave-one-source-out
The main experiment trains on three sources and tests on others. Here we rotate it: for each source that has both classes (Enron, Ling, SpamAssassin, CEAS_08), train on the **other three** and test on the one held out. We also report recall on Nazario and Nigerian, which are never in training. If a held-out score is much lower than the in-source score, the model depends on the style of its training sources.

In [8]:
all_rows = pd.concat([pd.read_csv(SPLITS / f'{n}.csv') for n in
                      ['train', 'val', 'test_insource', 'test_ceas', 'test_nazario', 'test_nigerian']],
                     ignore_index=True)
two_class = ['Enron', 'Ling', 'SpamAssassin', 'CEAS_08']
print(all_rows['source'].value_counts().to_dict())

loso_rows = []
for held in two_class:
    tr = all_rows[all_rows['source'].isin(two_class) & (all_rows['source'] != held)]
    te = all_rows[all_rows['source'] == held]
    v = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=100_000)
    m = LogisticRegression(C=best_C, max_iter=2000, random_state=SEED)
    m.fit(v.fit_transform(tr['text']), tr['label'])
    row = {'held_out': held, **metrics(te['label'], m.predict(v.transform(te['text'])))}
    for s in ['Nazario', 'Nigerian_Fraud']:
        sub = all_rows[all_rows['source'] == s]
        row[f'{s}_recall'] = m.predict(v.transform(sub['text'])).mean()
    loso_rows.append(row)
    print('finished', held)

loso = pd.DataFrame(loso_rows).round(4)
loso.to_csv(BASE / 'results' / 'tfidf_leave_one_source_out.csv', index=False)
loso

{'CEAS_08': 38748, 'Enron': 29364, 'SpamAssassin': 5553, 'Nigerian_Fraud': 3243, 'Ling': 2856, 'Nazario': 1544}
finished Enron
finished Ling
finished SpamAssassin
finished CEAS_08


,held_out,rows,recall,precision,f1,fpr,Nazario_recall,Nigerian_Fraud_recall
0,Enron,29364,0.8535,0.8683,0.8609,0.1168,0.2927,0.9630
1,Ling,2856,0.9236,0.8740,0.8981,0.0254,0.5900,0.9935
2,SpamAssassin,5553,0.9815,0.7319,0.8385,0.1286,0.5298,0.9917
3,CEAS_08,38748,0.8499,0.9203,0.8837,0.0915,0.8076,0.9932


## 9. Style-removal test
The sources differ in formatting (Enron and Ling are lowercased with spaces around punctuation, the others are not). Here we put **every email into one plain style**: lowercase, links replaced by `url`, numbers and years removed, punctuation removed. Then we retrain and retest. If scores drop a lot, the model was leaning on style or formatting. If they hold up, the scores come from the words themselves.

In [9]:
import re

def normalize(t):
    t = t.lower()
    t = re.sub(r'https?://\S+|www\.\S+', ' url ', t)   # links -> "url"
    t = re.sub(r'\d+', ' ', t)                          # numbers and years
    t = re.sub(r'[^a-z\s]', ' ', t)                     # punctuation and symbols
    return re.sub(r'\s+', ' ', t).strip()

vec_n = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=100_000)
Xn_train = vec_n.fit_transform(train['text'].map(normalize))
Xn_val = vec_n.transform(val['text'].map(normalize))

best_n = None
for C in [0.1, 1, 10, 100]:
    m = LogisticRegression(C=C, max_iter=2000, random_state=SEED).fit(Xn_train, train['label'])
    f1 = f1_score(val['label'], m.predict(Xn_val))
    print(f'C={C:<6} validation F1 = {f1:.4f}')
    if best_n is None or f1 > best_n[0]:
        best_n = (f1, C, m)
_, C_n, model_n = best_n
print('Best C (normalized):', C_n)

norm_rows = []
for name in TEST_FILES:
    df = pd.read_csv(SPLITS / f'{name}.csv')
    p = model_n.predict(vec_n.transform(df['text'].map(normalize)))
    norm_rows.append({'set': name, **metrics(df['label'], p)})
norm = pd.DataFrame(norm_rows)

cols = ['set', 'recall', 'precision', 'f1', 'fpr']
compare = (results[results['set'].isin(TEST_FILES)][cols]
           .merge(norm[cols], on='set', suffixes=('_original', '_normalized')).round(4))
compare.to_csv(BASE / 'results' / 'tfidf_normalized_vs_original.csv', index=False)

words_n = np.array(vec_n.get_feature_names_out())
print('Top phishing words after normalizing:', ', '.join(words_n[np.argsort(model_n.coef_[0])[::-1][:15]]))
compare

C=0.1    validation F1 = 0.9762
C=1      validation F1 = 0.9849
C=10     validation F1 = 0.9842
C=100    validation F1 = 0.9883
Best C (normalized): 100
Top phishing words after normalizing: http, your, money, you, remove, here, our, important, viagra, site, online, hello, quality, no, software


,set,recall_original,precision_original,f1_original,fpr_original,recall_normalized,precision_normalized,f1_normalized,fpr_normalized
0,test_insource,0.9912,0.9887,0.9899,0.0082,0.9899,0.9887,0.9893,0.0082
1,test_ceas,0.8500,0.9218,0.8845,0.0896,0.7755,0.9441,0.8515,0.0571
2,test_nazario,0.7902,NaN,NaN,NaN,0.7947,NaN,NaN,NaN
3,test_nigerian,0.9935,NaN,NaN,NaN,0.9926,NaN,NaN,NaN
4,test_nazario_paired,0.7902,0.8931,0.8385,0.0946,0.7947,0.9288,0.8565,0.0609


**Reading the tables:** in section 8, compare each held-out row to the in-source score from section 5. In section 9, compare the `_original` and `_normalized` columns. A large drop means style was helping the model; little change means the words themselves carry the signal. Either result is worth a sentence in the report.